<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0">

# Multi-Risk Derivatives Portfolios

The step from multi-risk derivatives instruments to multi-risk derivatives instrument portfolios is not a too large one. This part of the tutorial shows how to model an economy with three risk factors

In [ ]:
!git clone https://github.com/tpq-classes/computational_finance_dx.git
import sys
sys.path.append('computational_finance_dx')


In [ ]:
from dx import *
from pylab import plt
plt.style.use('seaborn-v0_8')

## Risk Factors

This sub-section models the single risk factors. We start with definition of the risk-neutral discounting object.

In [ ]:
# constant short rate
r = constant_short_rate('r', 0.02)

**Three risk factors** ares modeled:

* geometric Brownian motion
* jump diffusion
* stochastic volatility process

In [ ]:
# market environments
me_gbm = market_environment('gbm', dt.datetime(2020, 1, 1))
me_jd = market_environment('jd', dt.datetime(2020, 1, 1))
me_sv = market_environment('sv', dt.datetime(2020, 1, 1))

Assumptions for the `geometric_brownian_motion` object.

In [ ]:
# geometric Brownian motion
me_gbm.add_constant('initial_value', 36.)
me_gbm.add_constant('volatility', 0.2) 
me_gbm.add_constant('currency', 'EUR')
me_gbm.add_constant('model', 'gbm')

Assumptions for the `jump_diffusion` object.

In [ ]:
# jump diffusion
me_jd.add_constant('initial_value', 36.)
me_jd.add_constant('volatility', 0.2)
me_jd.add_constant('lambda', 0.5)
    # probability for jump p.a.
me_jd.add_constant('mu', -0.75)
    # expected jump size [%]
me_jd.add_constant('delta', 0.1)
    # volatility of jump
me_jd.add_constant('currency', 'EUR')
me_jd.add_constant('model', 'jd')

Assumptions for the `stochastic_volatility` object.

In [ ]:
# stochastic volatility model
me_sv.add_constant('initial_value', 36.)
me_sv.add_constant('volatility', 0.2)
me_sv.add_constant('vol_vol', 0.1)
me_sv.add_constant('kappa', 2.5)
me_sv.add_constant('theta', 0.4)
me_sv.add_constant('rho', -0.5)
me_sv.add_constant('currency', 'EUR')
me_sv.add_constant('model', 'sv')

Finally, the unifying valuation assumption for the **valuation environment**.

In [ ]:
# valuation environment
val_env = market_environment('val_env', dt.datetime(2020, 1, 1))
val_env.add_constant('paths', 10000)
val_env.add_constant('frequency', 'W')
val_env.add_curve('discount_curve', r)
val_env.add_constant('starting_date', dt.datetime(2020, 1, 1))
val_env.add_constant('final_date', dt.datetime(2020, 12, 31))

These are added to the single `market_environment` objects of the risk factors.

In [ ]:
# add valuation environment to market environments
me_gbm.add_environment(val_env)
me_jd.add_environment(val_env)
me_sv.add_environment(val_env)

Finally, the **market model** with the risk factors and the correlations between them.

In [ ]:
risk_factors = {'gbm' : me_gbm, 'jd' : me_jd, 'sv' : me_sv}
correlations = [['gbm', 'jd', 0.66], ['jd', 'sv', -0.75]]

## Derivatives

In this sub-section, we model the single derivatives instruments.

### American Put Option

The first derivative instrument is an **American put option**.

In [ ]:
gbm = geometric_brownian_motion('gbm_obj', me_gbm)

In [ ]:
me_put = market_environment('put', dt.datetime(2020, 1, 1))
me_put.add_constant('maturity', dt.datetime(2020, 12, 31))
me_put.add_constant('strike', 40.)
me_put.add_constant('currency', 'EUR')
me_put.add_environment(val_env)

In [ ]:
am_put = valuation_mcs_american_single('am_put', mar_env=me_put, underlying=gbm,
                       payoff_func='np.maximum(strike - instrument_values, 0)')

In [ ]:
am_put.present_value(fixed_seed=True, bf=5)

### European Maximum Call on 2 Assets

The second derivative instrument is a **European maximum call option on two risk factors**.

In [ ]:
jd = jump_diffusion('jd_obj', me_jd)

In [ ]:
me_max_call = market_environment('put', dt.datetime(2020, 1, 1))
me_max_call.add_constant('maturity', dt.datetime(2020, 9, 15))
me_max_call.add_constant('currency', 'EUR')
me_max_call.add_environment(val_env)

In [ ]:
payoff_call = "np.maximum(np.maximum(maturity_value['gbm'], maturity_value['jd']) - 34., 0)"

In [ ]:
assets = {'gbm' : me_gbm, 'jd' : me_jd}
asset_corr = [correlations[0]]

In [ ]:
asset_corr

In [ ]:
max_call = valuation_mcs_european_multi(
            'max_call',
             me_max_call,
             assets,
             asset_corr,
             payoff_func=payoff_call)

In [ ]:
max_call.present_value(fixed_seed=False)

In [ ]:
max_call.delta('jd')

In [ ]:
max_call.delta('gbm')

### American Minimum Put on 2 Assets

The third derivative instrument is an **American minimum put on two risk factors**.

In [ ]:
sv = stochastic_volatility('sv_obj', me_sv)

In [ ]:
me_min_put = market_environment('min_put', dt.datetime(2020, 1, 1))
me_min_put.add_constant('maturity', dt.datetime(2020, 6, 17))
me_min_put.add_constant('currency', 'EUR')
me_min_put.add_environment(val_env)

In [ ]:
payoff_put = "np.maximum(32. - np.minimum(instrument_values['jd'], instrument_values['sv']), 0)"

In [ ]:
assets = {'jd' : me_jd, 'sv' : me_sv}
asset_corr = [correlations[1]]
asset_corr

In [ ]:
min_put = valuation_mcs_american_multi(
                'min_put', val_env=me_min_put, risk_factors=assets,
                correlations=asset_corr, payoff_func=payoff_put)

In [ ]:
min_put.present_value(fixed_seed=True)

In [ ]:
min_put.delta('jd')

In [ ]:
min_put.delta('sv')

## Portfolio

To compose a derivatives portfolio, `derivatives_position` objects are needed.

In [ ]:
am_put_pos = derivatives_position(
                    name='am_put_pos',
                    quantity=2,
                    underlyings=['gbm'],
                    mar_env=me_put,
                    otype='American single',
                    payoff_func='np.maximum(instrument_values - 36., 0)')

In [ ]:
max_call_pos = derivatives_position(
                    'max_call_pos', 3, ['gbm', 'jd'],
                    me_max_call, 'European multi',
                    payoff_call)

In [ ]:
min_put_pos = derivatives_position(
                    'min_put_pos', 5, ['sv', 'jd'],
                    me_min_put, 'American multi',
                    payoff_put)

These objects are to be collected in `dictionary` objects.

In [ ]:
positions = {'am_put_pos' : am_put_pos, 'max_call_pos' : max_call_pos,
             'min_put_pos' : min_put_pos}

All is together to instantiate the `derivatives_portfolio` class.

In [ ]:
port = derivatives_portfolio(name='portfolio',
                             positions=positions,
                             val_env=val_env,
                             risk_factors=risk_factors,
                             correlations=correlations)

Let us have a look at the major **portfolio statistics**.

In [ ]:
%time stats = port.get_statistics()
stats

In [ ]:
stats['pos_value'].sum()

Finally, a graphical look at **two selected, simulated paths** of the stochastic volatility risk factor and the jump diffusion risk factor, respectively.

In [ ]:
path_no = 3
paths1 = port.underlying_objects['sv'].get_instrument_values()[:, path_no]
paths2 = port.underlying_objects['jd'].get_instrument_values()[:, path_no]

In [ ]:
paths1

In [ ]:
paths2

The resulting plot illustrates the strong **negative correlation**.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
plt.figure(figsize=(10, 6))
plt.plot(port.time_grid, paths1, 'r', label='sv')
plt.plot(port.time_grid, paths2, 'b', label='jd')
plt.gcf().autofmt_xdate()
plt.legend(loc=0); plt.grid(True)
# negatively correlated underlyings

**Copyright, License & Disclaimer**

© Dr. Yves J. Hilpisch | The Python Quants GmbH

DX Analytics (the "dx library" or "dx package") is licensed under the GNU Affero General
Public License version 3 or later (see http://www.gnu.org/licenses/).

DX Analytics comes with no representations or warranties, to the extent
permitted by applicable law.

http://tpq.io | [dx@tpq.io](mailto:team@tpq.io) |
http://twitter.com/dyjh

<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

**Quant Platform** | http://pqp.io

**Python for Finance Training** | http://training.tpq.io

**Certificate in Computational Finance** | http://compfinance.tpq.io

**Derivatives Analytics with Python (Wiley Finance)** |
http://dawp.tpq.io

**Python for Finance (2nd ed., O'Reilly)** |
http://py4fi.tpq.io